# 23 — Pull IDMC Internal Displacement Data

**Source:** Internal Displacement Monitoring Centre (IDMC)  
**Provider:** IDMC, Geneva  
**Access:** IDMC Helix API — free, no registration required  
**Coverage:** Global, 2008–present  
**Frequency:** Annual  

## What this notebook does
Pulls conflict-driven and disaster-driven internal displacement from the IDMC API.
IDP *flow velocity* (new displacements per year) is a leading indicator for conflict
escalation distinct from the UNHCR stock figures already in the pipeline.

## Features pulled

| Feature | Description |
|---|---|
| `conflict_new_displacements` | New IDPs due to conflict/violence (flow) |
| `conflict_stock_displacement` | Total IDPs from conflict at year-end (stock) |
| `disaster_new_displacements` | New IDPs due to natural disasters (flow) |
| `disaster_stock_displacement` | Total IDPs from disasters at year-end (stock) |
| `total_new_displacements` | Sum of conflict + disaster new displacements |

## Required environment variables
```
ADLS_ACCOUNT_NAME  — Azure storage account name
ADLS_CONTAINER     — Container name (default: 'data')
```

In [ ]:
import os
import requests
import pandas as pd
from datetime import datetime
from azure.identity import DefaultAzureCredential

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

IDMC_API = "https://helix-tools-api.idmcdb.org/external-api/idmc_displacement_all_dataset/"

print(f"ADLS target : abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/raw/idmc/")
print(f"Run date    : {RUN_DATE}")

## ADLS helper

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential": credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

## Pull all IDMC displacement records

In [ ]:
resp = requests.get(IDMC_API, params={"limit": 10000}, timeout=60)
resp.raise_for_status()
payload = resp.json()

# IDMC returns {"results": [...], "count": N}
records = payload.get("results", payload) if isinstance(payload, dict) else payload
df_raw = pd.DataFrame(records)
print(f"Raw records: {len(df_raw):,} rows")
print(f"Columns    : {list(df_raw.columns)}")
df_raw.head(3)

## Normalise column names and types

IDMC API field names differ slightly across releases — we map to stable output names.

In [ ]:
# Flexible column mapping — adapt if IDMC renames fields
COL_MAP = {
    # identifiers
    "iso3":                              "iso3",
    "country_name":                      "country",
    "year":                              "year",
    # conflict displacement
    "conflict_new_displacements":        "conflict_new_displacements",
    "conflict_stock_displacement":       "conflict_stock_displacement",
    # disaster displacement
    "disaster_new_displacements":        "disaster_new_displacements",
    "disaster_stock_displacement":       "disaster_stock_displacement",
}

available = {k: v for k, v in COL_MAP.items() if k in df_raw.columns}
missing   = [k for k in COL_MAP if k not in df_raw.columns]
if missing:
    print(f"Note: {len(missing)} expected columns absent from API response: {missing}")

df_idmc = df_raw[list(available.keys())].rename(columns=available).copy()
df_idmc["year"] = pd.to_numeric(df_idmc["year"], errors="coerce").astype("Int64")

for col in ["conflict_new_displacements", "conflict_stock_displacement",
            "disaster_new_displacements",  "disaster_stock_displacement"]:
    if col in df_idmc.columns:
        df_idmc[col] = pd.to_numeric(df_idmc[col], errors="coerce")

# Derived total
if "conflict_new_displacements" in df_idmc and "disaster_new_displacements" in df_idmc:
    df_idmc["total_new_displacements"] = (
        df_idmc["conflict_new_displacements"].fillna(0) +
        df_idmc["disaster_new_displacements"].fillna(0)
    )

df_idmc = df_idmc.sort_values(["iso3", "year"]).reset_index(drop=True)
print(f"Final shape: {df_idmc.shape}")
print(f"Countries  : {df_idmc['iso3'].nunique()}")
print(f"Year range : {df_idmc['year'].min()}–{df_idmc['year'].max()}")
df_idmc.head()

## Write to ADLS

In [ ]:
write_parquet(df_idmc, f"raw/idmc/{RUN_DATE}/idmc_panel.parquet")

## Summary

In [ ]:
print("=" * 55)
print("IDMC pull complete")
print("=" * 55)
print(f"  Rows (country-years) : {len(df_idmc):,}")
print(f"  Countries            : {df_idmc['iso3'].nunique()}")
print(f"  Year range           : {df_idmc['year'].min()}–{df_idmc['year'].max()}")
print()
print("ADLS path written:")
print(f"  raw/idmc/{RUN_DATE}/idmc_panel.parquet")